In [42]:
# !wget https://huggingface.co/1-800-BAD-CODE/sentence_boundary_detection_multilang/blob/main/sbd_49lang_bert_small.onnx .


In [43]:
import os
import urllib.request
import textwrap
from typing import List
import numpy as np
try:
    
    import onnxruntime as ort
    from sentencepiece import SentencePieceProcessor
except ImportError:
    print("Error: Required packages are missing. Please install them:")
    print("pip install onnxruntime sentencepiece numpy")
    exit(1)



In [44]:
onnx_model_path = "./models/sbd_49lang_bert_small.onnx"
sp_model_path = "./models/spe_mixed_case_64k_49lang.model"

print("\nLoading models...")
sp = SentencePieceProcessor(sp_model_path)
ort_session = ort.InferenceSession(onnx_model_path)

input_texts: List[str] = [
    "the new d.n.a. sample has been multiplexed, and the gametes are already dividing. let's get the c.p.d. over there. dinner's at 630 p.m. see that piece on you in the l.a. times? chicago p.d. will eat him alive.",
    "魔鬼兵團都死了？但是如果这让你不快乐就别做了。您就不能发个电报吗。我們都準備好了。",
    "él es uno de aquellos. ¿tiene algo de beber? cómo el aislamiento no vale la pena.",
    "พวกเขาต้องโกรธมากเลยใช่ไหม โทษทีนะลูกของเราไม่เป็นอะไรใช่ไหม ถึงเจ้าจะลากข้าไปเจ้าก็ไม่ได้อะไรอยู่ดี ผมคิดว่าจะดีกว่านะถ้าคุณไม่ออกไปไหน",
    "розігни і зігни, будь ласка. я знаю, ваши люди храбры. было приятно, правда? для начала, тебе нужен собственный свой самолет.",
    "szedłem tylko do. pamiętaj, nigdy się nie obawiaj żyć na krawędzi ryzyka. ćwiczę już od dwóch tygodni a byłem zabity tylko raz.",
]

threshold = 0.5
print("\nRunning inference...")
print("=" * 60)


Loading models...

Running inference...


In [ ]:
import fitz # The main module for PyMuPDF

def load_and_extract_text(pdf_path):
    """
    Opens a PDF file and extracts all text from it.

    Args:
        pdf_path: The path to the PDF file.
    """
    # Open the PDF document
    doc = fitz.open(pdf_path) #
    
    full_text = ""
    # Iterate over all pages in the document
    for page in doc:
        # Extract text from the page
        full_text += page.get_text() #
    
    # Close the document to free up resources
    doc.close()
    
    return full_text

# Example usage:
# text_content = load_and_extract_text("your_document.pdf")
# print(text_content)


In [52]:
text_pdf = load_and_extract_text("/home/ubuntu/data/NLP_Project/Sentence_Boundary/Lê Thành Hải - Thực hành Học Máy - Tập 1.pdf").replace("\n","").replace("  "," ").replace("•","")

In [56]:
def process_pdf_to_sentences(text_pdf_full, sp, ort_session, threshold=0.5):
    """
    Hàm chuẩn hóa Pipeline SBD: Nhận chuỗi siêu dài, cắt chunk gối nhau (Sliding Window), 
    và trả về danh sách các câu hoàn chỉnh.
    """
    # 1. Tokenize toàn bộ bài
    ids = sp.EncodeAsIds(text_pdf_full)
    
    # 2. Định nghĩa cấu hình Sliding Window
    MAX_LEN = 250   # Kích thước chunk lớn nhất
    OVERLAP = 20    # Gối lên nhau 20 token để không bị mất dấu ngắt câu ở mép
    
    global_probabilities = np.zeros(len(ids)) # Mảng lưu xác suất ngắt câu của TOÀN BỘ bài
    overlap_counts = np.zeros(len(ids))       # Biến đếm số lần 1 token được model dự đoán (để chia trung bình)
    
    # 3. Quét cửa sổ trượt qua toàn bộ bài
    for start_idx in range(0, len(ids), MAX_LEN - OVERLAP):
        end_idx = min(start_idx + MAX_LEN, len(ids))
        chunk_ids = ids[start_idx:end_idx]
        
        # Thêm BOS, EOS
        input_ids = [sp.bos_id()] + chunk_ids + [sp.eos_id()]
        input_tensor = np.array([input_ids], dtype=np.int64)
        
        # Build Inputs động
        onnx_inputs = {"input_ids": input_tensor}
        expected_inputs = [i.name for i in ort_session.get_inputs()]
        if "attention_mask" in expected_inputs:
            onnx_inputs["attention_mask"] = np.ones_like(input_tensor, dtype=np.int64)
        if "token_type_ids" in expected_inputs:
            onnx_inputs["token_type_ids"] = np.zeros_like(input_tensor, dtype=np.int64)
            
        # Inference
        outputs = ort_session.run(None, onnx_inputs)[0]
        chunk_probs = outputs[0, 1:-1] # Bỏ xác suất ở hai đầu BOS/EOS
        
        # Cộng dồn xác suất vào mảng Global
        global_probabilities[start_idx:end_idx] += chunk_probs
        overlap_counts[start_idx:end_idx] += 1
        
        # Dừng nếu đã quét đến token cuối cùng
        if end_idx == len(ids):
            break

    # 4. Tính xác suất ngắt câu trung bình (do điểm neo bị tính 2 lần vùng overlap)
    final_probs = global_probabilities / overlap_counts
    
    # 5. Lọc ranh giới câu bằng Threshold
    break_points = np.squeeze(np.argwhere(final_probs > threshold), axis=1).tolist()
    if type(break_points) is not list:
        break_points = [break_points]
        
    if not break_points or break_points[-1] != len(ids) - 1:
        break_points.append(len(ids) - 1)
        
    # 6. MẢNG OUTPUT: Decode tạo danh sách câu văn hoàn chỉnh
    sentences = []
    
    for i, bp in enumerate(break_points):
        start = 0 if i == 0 else (break_points[i - 1] + 1)
        # Giới hạn mảng để không vấp lỗi array index
        sub_ids = ids[start : bp + 1]
        if len(sub_ids) > 0:
            sub_text = sp.DecodeIds(sub_ids).strip()
            if sub_text: # Bỏ qua câu trống
                sentences.append(sub_text)
                
    return sentences

# --- SỬ DỤNG LÀM OUTPUT ---
# Truyền nội dung nguyên bản từ file PDF (không cần chia bằng split(".") nữa)
full_raw_text = text_pdf.replace("\n", " ").replace("  ", " ")
# Dùng hàm này trước khi xử lý
full_raw_text = clean_pdf_text(text_pdf)

# Đưa full_raw_text vào vòng lặp Chunking ở bước trước...
ids = sp.EncodeAsIds(full_raw_text)
# Hệ thống AI sẽ lo việc nhận diện dấu phẩy, dấu chấm giùm bạn!
final_sentences = process_pdf_to_sentences(full_raw_text, sp, ort_session, threshold=0.5)

print(f"✅ Đã trích xuất được {len(final_sentences)} câu.")
for idx, sentence in enumerate(final_sentences[:5]):
    print(f"Câu {idx + 1}: {sentence}")


✅ Đã trích xuất được 6292 câu.
Câu 1: DỰA THEOẤN BẢNLẦN TH ⁇ HAIAurélien GéronNXB THÔNG TINVÀ TRUYỀN THÔNGNHÓM D ⁇ CH THU ⁇ TMACHINE LEARNING CƠ BẢNKhái niệm,
Câu 2: Công cụ ⁇  Kỹthuật Xây dựngnhững Hệthống Thông minhKiến thức Nền tảngvềHọc MáyThực hành Học Máyvới Scikit ⁇ Learn,Keras  ⁇  TensorFlowT ⁇ P 1DỰA THEO ẤN BẢN LẦN TH ⁇ HAIThực hành Học Máy vớiScikit ⁇ Learn, Keras, vàTensorFlowKhái niệm, Công cụ ⁇  Kỹthuật Xây dựngnhững Hệthống Thông minhTập 1 ⁇  Kiến thức Nền tảng vềHọc MáyAurélien GéronThân gửi bạn Lê Thành Hải  ⁇ lethhai3003 ⁇ gmail.com ⁇ Cuốn sách này đã được bảo hộ, bản quyền thuộc vềMLBVN Group  ⁇  FUNiX.Thực hành Học Máy với Scikit ⁇ Learn, Keras, và TensorFlowTập 1  ⁇  Kiến thức Nền tảng vềHọc MáyAurélien GéronCopyright © 2019 Kiwisoft S.A.
Câu 3: S.
Câu 4: All rights reserved.
Câu 5: Published by O'Reilly Media, Inc.


In [54]:
import unicodedata

def clean_pdf_text(raw_text):
    # 1. Ép toàn bộ mã Unicode về chuẩn Dựng Sẵn (NFC) để tương thích với mọi mô hình AI.
    # Tính năng này sẽ tự động gộp chữ A + ^ + . thành chữ Ậ.
    text = unicodedata.normalize('NFC', raw_text)
    
    # 2. Xử lý các ký tự Typography đặc thủ của file PDF (nháy cong, gạch ngang siêu dài)
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    
    # 3. Dọn dẹp spacing
    text = text.replace("\n", " ").replace("  ", " ").strip()
    return text

# Dùng hàm này trước khi xử lý
full_raw_text = clean_pdf_text(text_pdf)

# Đưa full_raw_text vào vòng lặp Chunking ở bước trước...
ids = sp.EncodeAsIds(full_raw_text)
